In [ ]:
from google.colab import drive
import os
import pandas as pd
import zipfile

# Mount Google Drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# The path to the main directory containing all station folders
main_directory = '/content/drive/MyDrive/Datasets/Mesonet'

In [ ]:
# List of station folders
stations = ['ACME', 'BESS', 'BUTL', 'CHEY', 'CHIC', 'FTCB', 'HOBA', 'MINC', 'NINN', 'WASH', 'WEAT']

In [ ]:

# Function to read CSV files from a directory
def read_csv_files_from_directory(directory):
    csv_data = []
    for file in os.listdir(directory):
        if file.endswith('.csv'):
            file_path = os.path.join(directory, file)
            print(f"Reading CSV file: {file_path}")
            df = pd.read_csv(file_path)
            print(f"Columns in {file}: {df.columns}")
            csv_data.append(df)
    return csv_data

# Function to extract and read ZIP files from a directory
def extract_and_read_zip_files(directory):
    zip_data = []
    for file in os.listdir(directory):
        if file.endswith('.zip'):
            file_path = os.path.join(directory, file)
            with zipfile.ZipFile(file_path, 'r') as zip_ref:
                zip_ref.extractall(directory)
                for extracted_file in zip_ref.namelist():
                    if extracted_file.endswith('.csv'):
                        extracted_file_path = os.path.join(directory, extracted_file)
                        print(f"Reading extracted CSV file: {extracted_file_path}")
                        df = pd.read_csv(extracted_file_path)
                        print(f"Columns in {extracted_file}: {df.columns}")
                        zip_data.append(df)
    return zip_data

# Function to merge dataframes on 'TIME' column with suffixes to handle duplicate columns
def merge_dataframes_on_time(dataframes):
    if not dataframes:
        return pd.DataFrame()
    merged_df = dataframes[0]
    for df in dataframes[1:]:
        if 'TIME' not in df.columns:
            print("Warning: 'TIME' column not found in one of the dataframes")
            continue
        merged_df = pd.merge(merged_df, df, on='TIME', how='outer', suffixes=('', '_y'))
        # Remove duplicate columns
        merged_df = merged_df.loc[:, ~merged_df.columns.str.endswith('_y')]
    return merged_df

# Process each station folder
all_data = {}
for station in stations:
    station_directory = os.path.join(main_directory, station)
    csv_data = read_csv_files_from_directory(station_directory)
    zip_data = extract_and_read_zip_files(station_directory)
    all_data[station] = merge_dataframes_on_time(csv_data + zip_data)

# The all_data dictionary contains merged dataframes for each station
# Access merged data for the ACME  station
acme_data = all_data['ACME']
print(acme_data.head())


Reading CSV file: /content/drive/MyDrive/Datasets/Mesonet/ACME/ACME_07-08_Atmo (1).csv
Columns in ACME_07-08_Atmo (1).csv: Index(['STID', 'TIME', 'PRES', 'TAIR', 'TMIN', 'TMAX', 'TDEW', 'RELH', 'WDIR',
       'WSPD', 'WMAX', 'RAIN', 'SRAD'],
      dtype='object')
Reading CSV file: /content/drive/MyDrive/Datasets/Mesonet/ACME/ACME_08-09_Atmo.csv
Columns in ACME_08-09_Atmo.csv: Index(['STID', 'TIME', 'PRES', 'TAIR', 'TMIN', 'TMAX', 'TDEW', 'RELH', 'WDIR',
       'WSPD', 'WMAX', 'RAIN', 'SRAD'],
      dtype='object')
Reading CSV file: /content/drive/MyDrive/Datasets/Mesonet/ACME/ACME_09-10_Atmo (1).csv
Columns in ACME_09-10_Atmo (1).csv: Index(['STID', 'TIME', 'PRES', 'TAIR', 'TMIN', 'TMAX', 'TDEW', 'RELH', 'WDIR',
       'WSPD', 'WMAX', 'RAIN', 'SRAD'],
      dtype='object')
Reading CSV file: /content/drive/MyDrive/Datasets/Mesonet/ACME/ACME_10-11_Atmo.csv
Columns in ACME_10-11_Atmo.csv: Index(['STID', 'TIME', 'PRES', 'TAIR', 'TMIN', 'TMAX', 'TDEW', 'RELH', 'WDIR',
       'WSPD', 'WMAX',

In [ ]:
# Access merged data for the WASH  station
wash_data = all_data['WASH']
print(wash_data.head())

   STID              TIME    PRES  TAIR  TMIN  TMAX  TDEW  RELH  WDIR  WSPD  \
0  WASH  2006-12-31T23:00 -999.00  -999  -999  -999  -999  -999  -999  -999   
1  WASH  2007-01-01T00:00   29.02    32    31    32    26    79   308    11   
2  WASH  2007-01-01T01:00   29.02    30    30    31    26    82   315     9   
3  WASH  2007-01-01T02:00   29.04    30    29    30    26    85   314     8   
4  WASH  2007-01-01T03:00   29.04    29    29    30    25    85   329     9   

   WMAX   RAIN  SRAD  
0  -999 -999.0  -999  
1    17    0.0     0  
2    14    0.0     0  
3    13    0.0     0  
4    14    0.0     0  


In [ ]:
# Combine all merged dataframes into a single dataframe
combined_df = merge_dataframes_on_time(list(all_data.values()))

# access combined data
print(combined_df.head())


   STID              TIME    PRES   TAIR   TMIN   TMAX   TDEW   RELH   WDIR  \
0  ACME  2006-12-31T23:00 -999.00 -999.0 -999.0 -999.0 -999.0 -999.0 -999.0   
1  ACME  2007-01-01T00:00   28.86   32.0   31.0   32.0   25.0   75.0  326.0   
2  ACME  2007-01-01T01:00   28.86   30.0   30.0   31.0   25.0   79.0  320.0   
3  ACME  2007-01-01T02:00   28.87   29.0   29.0   30.0   25.0   83.0  321.0   
4  ACME  2007-01-01T03:00   28.88   29.0   28.0   29.0   25.0   85.0  327.0   

    WSPD   WMAX   RAIN   SRAD  
0 -999.0 -999.0 -999.0 -999.0  
1    9.0   13.0    0.0    0.0  
2    8.0   12.0    0.0    0.0  
3    8.0   13.0    0.0    0.0  
4    8.0   11.0    0.0    0.0  


In [ ]:
  combined_df.head()

,STID,TIME,PRES,TAIR,TMIN,TMAX,TDEW,RELH,WDIR,WSPD,WMAX,RAIN,SRAD
0,ACME,2006-12-31T23:00,-999.00,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0
1,ACME,2007-01-01T00:00,28.86,32.0,31.0,32.0,25.0,75.0,326.0,9.0,13.0,0.0,0.0
2,ACME,2007-01-01T01:00,28.86,30.0,30.0,31.0,25.0,79.0,320.0,8.0,12.0,0.0,0.0
3,ACME,2007-01-01T02:00,28.87,29.0,29.0,30.0,25.0,83.0,321.0,8.0,13.0,0.0,0.0
4,ACME,2007-01-01T03:00,28.88,29.0,28.0,29.0,25.0,85.0,327.0,8.0,11.0,0.0,0.0


In [ ]:
combined_df.tail()

,STID,TIME,PRES,TAIR,TMIN,TMAX,TDEW,RELH,WDIR,WSPD,WMAX,RAIN,SRAD
152901,NaN,2024-06-10T20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152902,NaN,2024-06-10T21:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152903,NaN,2024-06-10T22:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152904,NaN,2024-06-10T23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152905,NaN,2024-06-11T00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Drop rows with any NaN values
cleaned_df = combined_df.dropna()

# Display the last few rows of the cleaned DataFrame
cleaned_df.tail()


,STID,TIME,PRES,TAIR,TMIN,TMAX,TDEW,RELH,WDIR,WSPD,WMAX,RAIN,SRAD
8756,ACME,2007-12-31T19:00,28.96,36.0,34.0,39.0,14.0,41.0,321.0,8.0,14.0,0.0,0.0
8757,ACME,2007-12-31T20:00,29.00,35.0,33.0,36.0,15.0,43.0,320.0,6.0,12.0,0.0,0.0
8758,ACME,2007-12-31T21:00,29.03,33.0,31.0,34.0,16.0,50.0,302.0,5.0,7.0,0.0,0.0
8759,ACME,2007-12-31T22:00,29.05,31.0,30.0,31.0,17.0,56.0,318.0,5.0,9.0,0.0,0.0
8760,ACME,2007-12-31T23:00,29.08,32.0,31.0,33.0,18.0,56.0,344.0,10.0,15.0,0.0,0.0


In [ ]:
cleaned_df.shape

(8761, 13)

In [ ]:
# List unique stations in the STID column
unique_stations = combined_df['STID'].unique()

# Count the number of unique stations
num_unique_stations = len(unique_stations)

# Count the occurrences of each unique station
station_counts = combined_df['STID'].value_counts()

print("Unique Stations:", unique_stations)
print("Number of Unique Stations:", num_unique_stations)
print("Station Counts:\n", station_counts)


Unique Stations: ['ACME' nan]
Number of Unique Stations: 2
Station Counts:
 STID
ACME    8761
Name: count, dtype: int64
